In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv('/content/drive/Othercomputers/My PC/Documents/Drive/DeepLearning/NLP/train.txt', sep = ';', header=None, names = ['text','emotions'])

In [4]:
df.head()

,text,emotions
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [6]:
unique_emotions = df['emotions'].unique()
emotion_numbers = {}
i=0

for em in unique_emotions:
  emotion_numbers[em]= i
  i+=1
df['emotions'] = df['emotions'].map(emotion_numbers)

In [7]:
df['emotions']

,emotions
0,0
1,0
2,1
3,2
4,1
...,...
15995,0
15996,0
15997,5
15998,1


In [8]:
df['text'] = df['text'].apply( lambda x : x.lower())

In [9]:
import string

def remove_punctuations(text):
  return text.translate(str.maketrans('','',string.punctuation))

In [10]:
df['text'] = df['text'].apply(remove_punctuations)

In [11]:
import re

def remove_numbers(text):
  return re.sub(r'\d+', '', text)

df['text'] = df['text'].apply(remove_numbers)

In [12]:
import re

def remove_urls(text):
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

df['text'] = df['text'].apply(remove_urls)

In [14]:
def remove_emojis_and_special_chars(text):

    return ''.join(char for char in text if char.isalnum() or char.isspace())

df['text'] = df['text'].apply(remove_emojis_and_special_chars)

In [15]:
import nltk

In [16]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [18]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
stop_words = set(stopwords.words('english'))


In [20]:
def remove_stopwords(text):
    words = word_tokenize(text)
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return ' '.join(filtered_words)

In [21]:
nltk.download('punkt_tab', quiet=True)
df['text'] = df['text'].apply(remove_stopwords)

In [22]:
df['text']

,text
0,didnt feel humiliated
1,go feeling hopeless damned hopeful around some...
2,im grabbing minute post feel greedy wrong
3,ever feeling nostalgic fireplace know still pr...
4,feeling grouchy
...,...
15995,brief time beanbag said anna feel like beaten
15996,turning feel pathetic still waiting tables sub...
15997,feel strong good overall
15998,feel like rude comment im glad


In [25]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(df['text'], df['emotions'], test_size=0.20, random_state=42)

In [26]:
x_train

,text
676,refers course though cant help feeling somehow...
12113,im starting feel im suffering fatigue
7077,feel like probably would liked book little bit...
13005,really feel awkward
12123,im feeling little grumpy today lame weather te...
...,...
13418,love leave reader feeling confused slightly de...
5390,feel delicate
860,starting feel little stressed
15795,feel stressed tired worn shape neglected


In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [39]:
vectorizer = TfidfVectorizer()
x_train_tfidf = vectorizer.fit_transform(x_train)
x_test_tfidf = vectorizer.transform(x_test)
x_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 116049 stored elements and shape (12800, 13359)>

In [40]:
WOB = CountVectorizer()
x_train_count = WOB.fit_transform(x_train)
x_test_count = WOB.transform(x_test)
x_train_count

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 116049 stored elements and shape (12800, 13359)>

In [58]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

In [59]:
LR_model_tfidf = LogisticRegression(max_iter=1000)
LR_model_tfidf.fit(x_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [60]:
predict_lr_tfidf = LR_model_tfidf.predict(x_test_tfidf)

In [61]:
print(accuracy_score(y_test, predict_lr_tfidf))

0.8615625


In [50]:
NB_model_BOW = MultinomialNB()
NB_model_BOW.fit(x_train_count, y_train)

MultinomialNB()

In [62]:
predict_nb_BOW = NB_model_BOW.predict(x_test_count)

In [63]:
LR_model_BOW = LogisticRegression(max_iter=1000)
LR_model_BOW.fit(x_train_count, y_train)

LogisticRegression(max_iter=1000)

In [64]:
predict_lr_BOW = LR_model_BOW.predict(x_test_count)

In [65]:
print(accuracy_score(y_test, predict_lr_BOW))

0.88875


In [66]:
print(accuracy_score(y_test, predict_nb_BOW))

0.7678125
